# 里程碑 3：TTA（a/b 對調）與 temperature scaling 校準實驗 —— 5 folds

這是**實驗量測**用的 notebook，不重新訓練，只對已經存好的 5 個 fold 權重、各自在
自己的驗證集（`fold_indices(fold=N)` 切出來的 held-out 那份，訓練時從沒看過）上
量測四種組合的 log loss：

1. baseline —— 原始順序，沒有 TTA、沒有校準
2. TTA —— response_a/response_b 對調各推論一次，換回欄位對齊後在機率層級平均
3. temperature scaling —— **每個 fold 各自用自己的驗證集配溫度 T**（不是套用其他
   fold 配出來的常數——每個 fold 的驗證集對那個 fold 的模型來說才是 held-out，
   各自獨立配 T 才公平，套用別的 fold 配出來的 T 等於在「不是那個 fold 的
   held-out 資料」上驗證，會失去 held-out 的意義）
4. TTA + temperature —— 先把兩個順序的 logits 在 logits 層級平均，再對合併後的
   logits 配溫度

只在 fold 0 量過一次（1.08495 → 1.08112，改善 +0.00383）不夠：那份驗證集有可能
剛好對這個手法有利，5 個 fold 各自獨立量測同一個改善才有說服力。**但即使 5 個
fold 都一致改善，也只代表這不是單一切分的巧合，不代表效果量必然完全轉移到隱藏
測試集**——5 個 fold 共用同一套模型/訓練 recipe/資料分布，不是 5 個真正獨立的
實驗，這點在下面的結論裡會用比較保守的說法。

用 `kaggle kernels push` 上傳（見 notebooks/README.md 的模式）：`competition_sources`
掛這場競賽的資料（重建每個 fold 的驗證切分要用到 train.csv），`dataset_sources` 接
涵蓋全部 5 folds 的永久 Dataset（`fold0-4-checkpoints`，見 CLAUDE.md「正式權重只信
Dataset」），**關網路**（讀本地權重，不用連 HuggingFace Hub）。

In [ ]:
"""貼進 Kaggle Notebook 第一個 cell —— 由 scripts/gen_notebook_bootstrap.py 產生，不要手改。"""

import pathlib
import sys

_llmcls_files = {
    '__init__.py': r'''"""LLM Classification Finetuning — 共用工具模組。

刻意寫成可 import 的模組（而不是單一 notebook），因為訓練會在
Kaggle Notebook 或遠端 GPU 上跑，本機只負責資料處理、CV 切分與提交組裝。
同一份程式碼兩邊都能 import，路徑差異由 config.py 吸收。
"""

from llmcls.config import DATA_DIR, LABEL_COLS, OUTPUT_DIR
from llmcls.metrics import UNIFORM_LOGLOSS, log_loss

__all__ = ["DATA_DIR", "OUTPUT_DIR", "LABEL_COLS", "log_loss", "UNIFORM_LOGLOSS"]
''',
    'calibration.py': r'''"""事後校準：temperature scaling。

log loss 吃的是機率校準品質，不是準確率——不用重新訓練，只要在驗證集的 logits 上
配一個純量溫度 T，讓 softmax(logits / T) 更校準，套用到測試集的 logits 上即可。

純 numpy，不需要 torch，可以在本機測試（真正的 logits 由 llmcls/train.py 的
predict_logits() / predict_logits_with_model() 產生，那兩個需要 torch/transformers）。
"""

from __future__ import annotations

import warnings

import numpy as np

from llmcls.metrics import log_loss, softmax


def apply_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    """回傳 softmax(logits / temperature)。temperature > 1 會讓機率分佈變平滑
    （撫平過度自信），temperature < 1 會讓分佈更尖銳，temperature == 1 等於沒校準。
    """
    return softmax(np.asarray(logits) / temperature)


def fit_temperature(
    logits: np.ndarray,
    labels: np.ndarray,
    lo: float = 0.1,
    hi: float = 5.0,
    n_grid: int = 50,
    n_refine: int = 4,
) -> float:
    """在 [lo, hi] 網格搜尋、逐步細化，找出讓 log loss 最小的溫度 T。

    T 只有一個純量，網格搜尋 + 逐步細化就足夠穩定，不需要另外拉 scipy 依賴
    （sklearn 有牽帶到 scipy，但那是 transitive dependency，不想仰賴它）。
    """
    logits = np.asarray(logits)
    labels = np.asarray(labels)
    orig_lo, orig_hi = lo, hi
    best_t = 1.0
    cur_lo, cur_hi = lo, hi
    for _ in range(n_refine):
        candidates = np.linspace(cur_lo, cur_hi, n_grid)
        losses = [log_loss(labels, apply_temperature(logits, t)) for t in candidates]
        idx = int(np.argmin(losses))
        best_t = float(candidates[idx])
        span = (cur_hi - cur_lo) / n_grid
        cur_lo, cur_hi = max(1e-3, best_t - span), best_t + span

    if best_t <= orig_lo * 1.05 or best_t >= orig_hi * 0.95:
        warnings.warn(
            f"fit_temperature 找到的 T={best_t:.3f} 貼著搜尋邊界 [{orig_lo}, {orig_hi}]，"
            "可能沒收斂，先擴大 lo/hi 範圍再看一次"
        )
    return best_t
''',
    'config.py': r'''"""路徑與常數。本機 / Kaggle Notebook 的差異全部集中在這裡。"""

from __future__ import annotations

import os
from pathlib import Path

COMPETITION = "llm-classification-finetuning"

# Kaggle Notebook 內資料掛載路徑：透過網頁 UI「Add Data」掛的話是
# /kaggle/input/<competition>/，但實測透過 `kaggle kernels push`（kernel-metadata.json
# 的 competition_sources）掛的話，實際掛在 /kaggle/input/competitions/<competition>/，
# 多一層 competitions/ —— 兩條路徑都要認，不要假設只有一種。
# 可用環境變數 LLMCLS_DATA_DIR 覆寫（例如指到 data/fixture 跑煙霧測試）。
_KAGGLE_INPUT_CANDIDATES = [
    Path("/kaggle/input") / COMPETITION,
    Path("/kaggle/input/competitions") / COMPETITION,
]
# 本競賽目錄 competitions/<slug>/，不是 git repo 根目錄 —— 工作區還有其他競賽。
_COMP_ROOT = Path(__file__).resolve().parents[2]


def _resolve_data_dir() -> Path:
    if env := os.environ.get("LLMCLS_DATA_DIR"):
        return Path(env)
    for candidate in _KAGGLE_INPUT_CANDIDATES:
        if candidate.exists():
            return candidate
    return _COMP_ROOT / "data"


DATA_DIR = _resolve_data_dir()
# Kaggle Notebook 只有 /kaggle/working 可寫。
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else _COMP_ROOT / "outputs"

# 訓練與推論是兩個獨立的 Kaggle Notebook：訓練 notebook 把權重存到 OUTPUT_DIR/model，
# Save Version 後那個資料夾變成一個 Kaggle Dataset，掛進推論 notebook 時的掛載路徑
# 由使用者在 Kaggle UI 上決定、無法預先得知，所以用環境變數覆寫（呼應 LLMCLS_DATA_DIR）。
MODEL_DIR = Path(os.environ.get("LLMCLS_MODEL_DIR", str(OUTPUT_DIR / "model")))

# 訓練用的 base model；推論 notebook 離線，權重從 MODEL_DIR 讀，不會連到這個 hub id。
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 512

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"

# 三分類的目標欄位，順序即 class 0/1/2，全專案共用這個順序。
LABEL_COLS = ["winner_model_a", "winner_model_b", "winner_tie"]
N_CLASSES = len(LABEL_COLS)

# 需要 parse 的 JSON 字串欄位（多輪對話存成 list of str）。
TEXT_COLS = ["prompt", "response_a", "response_b"]

SEED = 42
N_FOLDS = 5
''',
    'cv.py': r'''"""交叉驗證切分。

重點：訓練集裡有數千筆重複的 prompt。如果同一個 prompt 同時落在 train 和 valid
fold，本地分數會虛高、跟 LB 對不上。所以一律以 prompt 當 group 切分。
"""

from __future__ import annotations

import hashlib

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from llmcls.config import N_FOLDS, SEED


def prompt_group_key(df: pd.DataFrame) -> pd.Series:
    """以 prompt 文字的 hash 當 group id（正規化空白後再 hash）。"""
    normalized = df["prompt_text"].fillna("").str.strip().str.replace(r"\s+", " ", regex=True)
    return normalized.map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest())


def add_folds(
    df: pd.DataFrame, n_folds: int = N_FOLDS, seed: int = SEED, col: str = "fold"
) -> pd.DataFrame:
    """加上 fold 欄位：以 prompt 分組、以 label 分層。回傳新的 DataFrame。"""
    df = df.copy()
    groups = prompt_group_key(df)
    n_groups = groups.nunique()
    if n_groups < n_folds:
        raise ValueError(f"唯一 prompt 數 ({n_groups}) 少於 fold 數 ({n_folds})，無法切分")

    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df[col] = -1
    for fold, (_, valid_idx) in enumerate(splitter.split(df, df["label"], groups)):
        df.iloc[valid_idx, df.columns.get_loc(col)] = fold

    assert (df[col] >= 0).all(), "有列沒有被分配到 fold"
    _assert_no_group_leak(df, groups, col)
    return df


def _assert_no_group_leak(df: pd.DataFrame, groups: pd.Series, col: str) -> None:
    """驗證每個 prompt group 只出現在單一 fold —— 這是切分的重點，值得直接斷言。"""
    per_group = pd.DataFrame({"group": groups.to_numpy(), "fold": df[col].to_numpy()})
    leaked = per_group.groupby("group")["fold"].nunique()
    n_leaked = int((leaked > 1).sum())
    if n_leaked:
        raise AssertionError(f"有 {n_leaked} 個 prompt group 跨越多個 fold，切分有誤")


def fold_indices(df: pd.DataFrame, fold: int, col: str = "fold") -> tuple[np.ndarray, np.ndarray]:
    """回傳 (train_idx, valid_idx) 的位置索引。"""
    is_valid = (df[col] == fold).to_numpy()
    return np.flatnonzero(~is_valid), np.flatnonzero(is_valid)
''',
    'data.py': r'''"""資料載入與欄位解析。

注意：prompt / response_a / response_b 在原始 CSV 裡是 **JSON 編碼的字串陣列**
（多輪對話，每個 element 是一輪），不是純文字。這點在真實資料下載前尚未實地驗證，
所以 parse 採防禦式寫法：json.loads 失敗就退回當成單輪純文字。
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd

from llmcls.config import LABEL_COLS, TEST_CSV, TEXT_COLS, TRAIN_CSV

TURN_SEP = "\n\n"

# 退回純文字的比例超過這個門檻就直接報錯 —— 代表該欄根本不是 JSON，schema 假設錯了。
FALLBACK_ERROR_RATIO = 0.01


def _is_missing(raw: object) -> bool:
    """型別無關的缺值判斷（None / float nan / pd.NA 都算）。"""
    if raw is None:
        return True
    try:
        return bool(pd.isna(raw))
    except (TypeError, ValueError):
        # 例如 list、ndarray：pd.isna 回傳陣列或直接拋錯，都當作非缺值。
        return False


def _parse_turns_flagged(raw: object) -> tuple[list[str], bool]:
    """回傳 (turns, 是否退回純文字)。退回的次數會被上層統計，不能靜默吞掉。"""
    if _is_missing(raw):
        return [], False
    if isinstance(raw, list):
        return [("" if t is None else str(t)) for t in raw], False
    text = str(raw)
    try:
        parsed = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        # 不是合法 JSON —— 當成單輪純文字，避免整批資料因為個別壞格式而中斷。
        return [text], True
    if isinstance(parsed, list):
        return [("" if t is None else str(t)) for t in parsed], False
    return [str(parsed)], True


def parse_turns(raw: object) -> list[str]:
    """把一格 JSON 字串解析成 list[str]；缺值 / 解析失敗都不會炸掉。"""
    return _parse_turns_flagged(raw)[0]


def join_turns(turns: list[str]) -> str:
    return TURN_SEP.join(t for t in turns if t)


def _require_columns(df: pd.DataFrame, cols: list[str], source: Path) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{source} 缺少預期欄位 {missing}；實際欄位為 {list(df.columns)}。"
            " 若官方 schema 有變動，請同步更新 llmcls/config.py。"
        )


def _add_parsed_text(df: pd.DataFrame) -> pd.DataFrame:
    """解析文字欄位，並統計有多少列走了「退回純文字」的退路。

    這個計數是 schema 假設是否成立的第一手診斷：少數幾列是雜訊，比例一高就代表
    該欄根本不是 JSON。絕對不能靜默退回，否則模型會拿著原始 JSON 字串在訓練。
    """
    for col in TEXT_COLS:
        flagged = df[col].map(_parse_turns_flagged)
        turns = flagged.map(lambda x: x[0])
        n_fallback = int(flagged.map(lambda x: x[1]).sum())
        if n_fallback:
            msg = f"{col}：{n_fallback}/{len(df)} 列無法解析為 JSON 陣列，已退回單輪純文字"
            if n_fallback > FALLBACK_ERROR_RATIO * len(df):
                raise ValueError(
                    f"{msg} —— 比例過高，該欄的 schema 可能與預期不符。"
                    " 請檢查實際資料格式並更新 llmcls/data.py 的解析邏輯。"
                )
            warnings.warn(msg, stacklevel=2)
        df[f"{col}_turns"] = turns
        df[f"{col}_text"] = turns.map(join_turns)
    return df


def load_train(path: Path | None = None) -> pd.DataFrame:
    """載入訓練集，附上解析後的文字欄位與整數 label。"""
    path = Path(path) if path is not None else TRAIN_CSV
    if not path.exists():
        raise FileNotFoundError(f"找不到 {path}；請先執行 scripts/download_data.sh 下載競賽資料。")
    df = pd.read_csv(path)
    _require_columns(df, ["id", *TEXT_COLS, *LABEL_COLS], path)

    onehot = df[LABEL_COLS].to_numpy()
    bad = onehot.sum(axis=1) != 1
    if bad.any():
        raise ValueError(
            f"{path} 有 {int(bad.sum())} 列的 {LABEL_COLS} 不是恰好一個 1，"
            " 無法轉成單一 label，請檢查資料。"
        )
    df["label"] = onehot.argmax(axis=1)
    return _add_parsed_text(df)


def load_test(path: Path | None = None) -> pd.DataFrame:
    """載入測試集（沒有 label 欄）。"""
    path = Path(path) if path is not None else TEST_CSV
    if not path.exists():
        raise FileNotFoundError(f"找不到 {path}；請先執行 scripts/download_data.sh 下載競賽資料。")
    df = pd.read_csv(path)
    _require_columns(df, ["id", *TEXT_COLS], path)
    return _add_parsed_text(df)
''',
    'metrics.py': r'''"""評分指標。競賽用 multi-class log loss。"""

from __future__ import annotations

import numpy as np

from llmcls.config import N_CLASSES

# 均勻亂猜的分數 = ln(3) ≈ 1.0986。任何模型沒打敗這條線就等於沒有資訊量。
UNIFORM_LOGLOSS = float(np.log(N_CLASSES))

EPS = 1e-15


def softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


def log_loss(y_true: np.ndarray, y_prob: np.ndarray, eps: float = EPS) -> float:
    """multi-class log loss。y_true 是整數 label，y_prob 是 (n, n_classes) 機率。

    先 clip 到 [eps, 1-eps] 再逐列重新正規化。Kaggle 端的實際實作未經查證，
    但只要機率沒有極端到觸及 eps，各種變體的差異可忽略。
    """
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if y_prob.ndim != 2 or y_prob.shape[1] != N_CLASSES:
        raise ValueError(f"y_prob 形狀應為 (n, {N_CLASSES})，實際為 {y_prob.shape}")
    if len(y_true) != len(y_prob):
        raise ValueError(f"y_true ({len(y_true)}) 與 y_prob ({len(y_prob)}) 長度不一致")
    if not np.isfinite(y_prob).all():
        raise ValueError("y_prob 含有 NaN 或 inf")

    p = np.clip(y_prob, eps, 1 - eps)
    p = p / p.sum(axis=1, keepdims=True)
    return float(-np.mean(np.log(p[np.arange(len(y_true)), y_true])))
''',
    'submission.py': r'''"""提交檔組裝與驗證。

Kaggle 只會告訴你「submission 格式錯誤」，不會告訴你錯在哪裡，
所以在本機就把能檢查的都檢查掉。
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from llmcls.config import LABEL_COLS, N_CLASSES, OUTPUT_DIR, SAMPLE_SUBMISSION_CSV


def build_submission(ids: pd.Series | np.ndarray, probs: np.ndarray) -> pd.DataFrame:
    probs = np.asarray(probs, dtype=float)
    if probs.ndim != 2 or probs.shape[1] != N_CLASSES:
        raise ValueError(f"probs 形狀應為 (n, {N_CLASSES})，實際為 {probs.shape}")
    if len(ids) != len(probs):
        raise ValueError(f"ids ({len(ids)}) 與 probs ({len(probs)}) 長度不一致")
    sub = pd.DataFrame({"id": np.asarray(ids)})
    sub[LABEL_COLS] = probs
    return sub


def validate_submission(sub: pd.DataFrame, sample_path: Path | None = None) -> None:
    """檢查欄位、數值範圍、機率和；有 sample_submission 時再比對 id 集合。"""
    expected_cols = ["id", *LABEL_COLS]
    if list(sub.columns) != expected_cols:
        raise ValueError(f"欄位應為 {expected_cols}，實際為 {list(sub.columns)}")

    probs = sub[LABEL_COLS].to_numpy(dtype=float)
    if not np.isfinite(probs).all():
        raise ValueError("提交檔含有 NaN 或 inf")
    if (probs < 0).any() or (probs > 1).any():
        raise ValueError("機率值超出 [0, 1] 範圍")
    row_sums = probs.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-6):
        worst = float(np.abs(row_sums - 1.0).max())
        raise ValueError(f"每列機率和必須為 1，最大偏差 {worst:.2e}")
    if sub["id"].duplicated().any():
        raise ValueError("提交檔有重複的 id")

    sample_path = Path(sample_path) if sample_path is not None else SAMPLE_SUBMISSION_CSV
    if sample_path.exists():
        expected_ids = set(pd.read_csv(sample_path)["id"])
        actual_ids = set(sub["id"])
        if expected_ids != actual_ids:
            raise ValueError(
                f"id 集合與 sample_submission 不符："
                f"缺少 {len(expected_ids - actual_ids)} 筆、多出 {len(actual_ids - expected_ids)} 筆"
            )


def save_submission(
    sub: pd.DataFrame, name: str = "submission.csv", sample_path: Path | None = None
) -> Path:
    validate_submission(sub, sample_path)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / name
    sub.to_csv(path, index=False)
    return path
''',
    'text.py': r'''"""把 prompt / response_a / response_b 的 token id 組成單一模型輸入序列，
超過 max_len 時做 head+tail 截斷。

刻意在 id 層級操作（呼叫端先分別 encode 三段，這裡只管截斷與長度預算），而不是
組字串再整段重新 tokenize —— 這樣三段可以各自截斷，不會因為 response_a 太長就把
response_b 擠到只剩尾巴一小段，兩邊被模型看到的資訊量比較公平。

不依賴 torch / transformers，純 Python 可在本機測試。真正呼叫 tokenizer 的地方在
llmcls/train.py（那裡才需要 GPU 環境）。
"""

from __future__ import annotations

# 對應 [CLS] prompt [SEP] response_a [SEP] response_b [SEP] 的組法：1 個 CLS + 3 個 SEP。
NUM_SPECIAL_TOKENS = 4

# 三段的預算比例：prompt 通常比兩份回覆短，兩份回覆同等重要所以各拿更多。
DEFAULT_RATIOS = (0.2, 0.4, 0.4)


def truncate_ids(ids: list[int], budget: int, head_ratio: float = 0.5) -> list[int]:
    """留頭尾、砍中間。budget <= 0 回傳空列表，不需要截斷時原樣回傳。"""
    if budget <= 0:
        return []
    if len(ids) <= budget:
        return ids
    head_len = min(budget, max(1, round(budget * head_ratio)))
    tail_len = budget - head_len
    if tail_len <= 0:
        return ids[:head_len]
    return ids[:head_len] + ids[len(ids) - tail_len :]


def split_budget(total: int, ratios: tuple[float, float, float] = DEFAULT_RATIOS) -> tuple[int, int, int]:
    """依 ratios 把 total 分給三段；四捨五入的誤差全部歸給最後一段，確保三段總和精確等於 total。"""
    if total <= 0:
        return (0, 0, 0)
    a = int(total * ratios[0])
    b = int(total * ratios[1])
    c = total - a - b
    return (a, b, c)


def swap_ab_label(label: int) -> int:
    """訓練時 a/b 對調增強用：response_a/response_b 對調之後，原本「a 贏」
    （0）要變成「b 贏」（1），「b 贏」要變成「a 贏」，「打平」（2）不受影響
    ——這兩類是靠標籤本身的整數編碼互換位置，跟 llmcls.train.swap_ab() 只換
    DataFrame 欄位、不動標籤是兩回事（那個是給推論 TTA 用的，模型輸出的機率
    欄位事後靠 [:, [1, 0, 2]] 換回來對齊，不需要動標籤）。
    """
    return 1 - label if label in (0, 1) else label


def build_input_ids(
    prompt_ids: list[int],
    response_a_ids: list[int],
    response_b_ids: list[int],
    max_len: int,
    ratios: tuple[float, float, float] = DEFAULT_RATIOS,
    head_ratio: float = 0.5,
) -> tuple[list[int], list[int], list[int]]:
    """回傳截斷後的 (prompt_ids, response_a_ids, response_b_ids)。

    呼叫端還要自己補上 CLS/SEP 特殊 token，所以保證
    len(p) + len(a) + len(b) + NUM_SPECIAL_TOKENS <= max_len。
    """
    budget = max_len - NUM_SPECIAL_TOKENS
    if budget <= 0:
        raise ValueError(f"max_len ({max_len}) 太小，容不下 {NUM_SPECIAL_TOKENS} 個特殊 token")
    p_budget, a_budget, b_budget = split_budget(budget, ratios)
    return (
        truncate_ids(prompt_ids, p_budget, head_ratio),
        truncate_ids(response_a_ids, a_budget, head_ratio),
        truncate_ids(response_b_ids, b_budget, head_ratio),
    )
''',
    'train.py': r'''"""DeBERTa-v3-base 三分類微調：資料集組裝、訓練、推論。

只能在有 torch / transformers 的環境 import（Kaggle Notebook 或有 GPU 的機器）；
本機沒有 GPU，這個檔案沒有、也無法有本機測試覆蓋。截斷邏輯本身在 llmcls/text.py
裡用純 Python 測試過，這裡只是把它接上真正的 tokenizer 和 HF Trainer。

`train_fold()` 是核心入口，同時給兩種呼叫方式用：
- CLI：scripts/train.py（適合遠端 GPU，例如 RunPod）
- Kaggle Notebook cell：`from llmcls.train import train_fold` 直接呼叫
"""

from __future__ import annotations

import inspect
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

from llmcls.config import MAX_LEN, MODEL_DIR, MODEL_NAME, N_CLASSES, N_FOLDS, SEED
from llmcls.cv import add_folds, fold_indices
from llmcls.data import load_test, load_train
from llmcls.metrics import UNIFORM_LOGLOSS, log_loss, softmax
from llmcls.submission import build_submission, save_submission
from llmcls.text import build_input_ids, swap_ab_label
from llmcls.training_safety import should_stop_for_nonfinite
from llmcls.tta import average_swapped


class PreferenceDataset(torch.utils.data.Dataset):
    """把三欄文字 tokenize 成單一序列：[CLS] prompt [SEP] response_a [SEP] response_b [SEP]。"""

    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_len: int,
        labels: np.ndarray | None,
        ab_swap_prob: float = 0.0,
    ):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
        self.ab_swap_prob = ab_swap_prob
        # 訓練時 a/b 對調增強（milestone 3）：獨立的 RNG，不動 Python/numpy 的全域
        # 亂數狀態——Trainer 自己的資料洗牌也是靠全域亂數狀態，兩邊共用同一個全域
        # 產生器的話，開不開這個選項會連帶改變洗牌順序，A/B 比較就不乾淨了。固定
        # 用 SEED 起始，同一個 seed 重跑兩次會抽到同一組對調決定。
        self._rng = np.random.default_rng(SEED) if ab_swap_prob > 0 else None
        # 預先 encode 三欄一次，__getitem__ 只做截斷 + 拼接，不用每個 epoch 重複 tokenize。
        # 整批呼叫 tokenizer(...)，不要逐列呼叫 tokenizer.encode()——fast tokenizer 的
        # 平行化是在批次呼叫內部做的（Rust 那邊自己開執行緒），逐列呼叫每次都要重新付一次
        # Python↔Rust FFI 的固定開銷，資料量上萬列時這筆開銷不能忽略，TTA 又是同一批文字
        # 重新 tokenize 兩次（正常順序 + 對調順序），批次呼叫能把這個成本壓下去。
        self.prompt_ids = tokenizer(df["prompt_text"].tolist(), add_special_tokens=False)["input_ids"]
        self.response_a_ids = tokenizer(df["response_a_text"].tolist(), add_special_tokens=False)["input_ids"]
        self.response_b_ids = tokenizer(df["response_b_text"].tolist(), add_special_tokens=False)["input_ids"]

    def __len__(self) -> int:
        return len(self.prompt_ids)

    def __getitem__(self, idx: int) -> dict:
        a_ids, b_ids = self.response_a_ids[idx], self.response_b_ids[idx]
        label = int(self.labels[idx]) if self.labels is not None else None
        # 每次被抓取都重新擲一次骰子（不是固定對調某一半資料）——同一列資料在不同
        # epoch 可能拿到不同順序，訓練久了每一列平均都看過兩種順序。只有訓練集會
        # 傳非 0 的 ab_swap_prob，驗證集固定用原始順序，score 才能跟沒開這個選項
        # 的訓練直接比較。
        if self._rng is not None and self._rng.random() < self.ab_swap_prob:
            a_ids, b_ids = b_ids, a_ids
            if label is not None:
                label = swap_ab_label(label)
        p, a, b = build_input_ids(self.prompt_ids[idx], a_ids, b_ids, self.max_len)
        cls_id, sep_id = self.tokenizer.cls_token_id, self.tokenizer.sep_token_id
        input_ids = [cls_id, *p, sep_id, *a, sep_id, *b, sep_id]
        item = {"input_ids": input_ids, "attention_mask": [1] * len(input_ids)}
        if label is not None:
            item["labels"] = label
        return item


class StopOnNonFiniteLoss(TrainerCallback):
    """DeBERTa-v3 在 fp32、peak LR 附近實測會突然發散：loss 衝高、grad_norm 變 NaN，
    之後每一步都是壞的，權重永久壞掉但 Trainer 完全不知道、還是把剩下的 epoch 跑完
    （實測浪費了 83 分鐘 GPU 時間裡的 70 分鐘）。這裡一偵測到就叫它停，把剩下的時間
    省下來，`triggered` 讓呼叫端知道這次訓練發散過、權重不可信。

    只看單一次 log 的 grad_norm 曾經誤判過：fp16 下 GradScaler 遇到某一步梯度
    溢位會自動跳過那次更新、調低 scale factor 再繼續，這是正常現象，loss 本身
    仍然健康，不代表訓練壞掉——實測 group_by_length 那次「發散」在觸發停止的
    那一行 loss 是 1.079（跟前面每一步一樣正常），只有 grad_norm 是 inf，判定
    「有害」其實是這個過度敏感的舊邏輯誤觸發。真正的判斷邏輯（loss 非有限值
    立刻停；grad_norm 非有限值要連續 `grad_norm_patience` 次才算真的卡住）在
    `llmcls.training_safety.should_stop_for_nonfinite()`，本機有測試覆蓋。
    """

    def __init__(self, grad_norm_patience: int = 3) -> None:
        self.triggered = False
        self.grad_norm_patience = grad_norm_patience
        self._consecutive_nonfinite_grad_norm = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            should_stop, self._consecutive_nonfinite_grad_norm = should_stop_for_nonfinite(
                logs, self._consecutive_nonfinite_grad_norm, self.grad_norm_patience
            )
            if should_stop:
                control.should_training_stop = True
                self.triggered = True
        return control


# log_loss 不可能自然達到的高值，訓練中途的 log 看到這個數字就知道那次週期性
# 評估算在非有限值上，是壞掉的資料點，不是真的分數。
_NONFINITE_SENTINEL = 99.0


def _compute_metrics(eval_pred) -> dict:
    """log_loss() 對非法機率值嚴格報錯（這是它的正確行為，見 metrics.py）；但這裡是
    Trainer 的 eval callback，一炸整個 run 就白跑、連權重都存不到，所以不能用 raise。

    早期版本把非有限值夾到均勻機率再算分——結果發散的 checkpoint 剛好算出
    log loss = ln(3)，跟真的健康、剛好打平基準的 checkpoint 混在一起分不出來。
    改成回報一個大到不可能自然出現的哨兵值，訓練中途的 log 一看就知道那個點壞了。
    """
    logits, labels = eval_pred
    probs = softmax(np.asarray(logits))
    finite = np.isfinite(probs).all(axis=1)
    n_nonfinite = int((~finite).sum())
    if n_nonfinite:
        return {"log_loss": _NONFINITE_SENTINEL, "vs_uniform": float("nan"), "n_nonfinite": n_nonfinite}
    score = log_loss(labels, probs)
    return {"log_loss": score, "vs_uniform": UNIFORM_LOGLOSS - score, "n_nonfinite": n_nonfinite}


def _training_args(
    output_dir: Path,
    epochs: int,
    batch_size: int,
    lr: float,
    lr_scheduler_type: str = "linear",
    max_steps: int | None = None,
    eval_steps: int | None = None,
    fp16: bool = True,
    label_smoothing: float = 0.0,
    group_by_length: bool = False,
) -> TrainingArguments:
    # transformers 把 evaluation_strategy 改名成 eval_strategy 過；Kaggle Notebook 內建的
    # 版本不固定，用 inspect 挑對的參數名比硬編一個更穩。
    params = inspect.signature(TrainingArguments.__init__).parameters
    strategy_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
    # 一律用 steps（不是 epoch）當 eval/save 的節奏，完整訓練也一樣 —— 中途的週期性
    # 存檔是拿來在 kernel 中途出狀況時當復原點用的，1500 是給完整訓練用的預設值：
    # 跟 eval_subset_rows 搭配（train_fold 會把訓練中途的評估換成子集），拉開頻率
    # 不會犧牲發散偵測 —— 那是每 50 步看 loss/grad_norm 的 StopOnNonFiniteLoss
    # callback 在管，跟這裡的 eval 節奏無關。
    #
    # 不用 load_best_model_at_end：實測踩到一個問題——同樣的設定重跑兩次，
    # `load_best_model_at_end` 靠訓練中途對 eval_subset_rows（2000 筆）子集算出來
    # 的分數去挑「最佳」checkpoint，這個子集本身雜訊就不小，兩次重跑各自挑到不同
    # 進度的 checkpoint 當最終權重（實測分別挑中 epoch 1.049 跟 epoch 1.574），
    # 光是這個選擇上的雜訊就足以讓兩次「應該一樣」的最終分數飄動超過 0.01——跟
    # label smoothing/a、b 對調增強量到的效果量同一個量級，會讓 A/B 比較失去意義。
    # 子集分數只拿來在訓練中途看趨勢（原本的設計目的），不該拿來決定「用哪個版本
    # 的權重」；固定用訓練跑完當下的最終狀態，才不會多引入這層雜訊。
    default_eval_steps = max(1, max_steps // 2) if max_steps is not None else 1500
    steps = eval_steps or default_eval_steps
    kwargs = dict(
        output_dir=str(output_dir),
        **{strategy_key: "steps"},
        save_strategy="steps",
        eval_steps=steps,
        save_steps=steps,
        save_total_limit=1,
        load_best_model_at_end=False,
        learning_rate=lr,
        lr_scheduler_type=lr_scheduler_type,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        num_train_epochs=epochs,
        warmup_ratio=0.1,
        weight_decay=0.01,
        # 第一次踩到的坑是模型「裸」用 fp16（checkpoint 原本就存 fp16，from_pretrained
        # 沒有轉型），完全沒有 loss scaler 保護。train_fold() 現在會先強制 model.float()
        # 轉成真正的 fp32，這裡的 fp16=True 才是正規流程：autocast 動態轉型 + GradScaler
        # 做 loss scaling，梯度真的非有限值時 GradScaler 會跳過那一步而不是把權重弄壞，
        # StopOnNonFiniteLoss 則是最後一道防線。T4 有 fp16 tensor core，這樣才吃得到
        # 混合精度的加速。
        fp16=fp16,
        # Trainer 內建的 label smoothing：labels 還是整數類別（不用先轉成 one-hot），
        # HF 的 LabelSmoother 會在算 cross entropy 時自動把目標機率從 1.0 壓低、
        # 分一點出去給另外兩類。0.0 等於關閉，行為跟原本完全一樣。
        label_smoothing_factor=label_smoothing,
        # 把長度相近的樣本分到同一個 batch，減少 padding 浪費的算力。PreferenceDataset
        # 不是 datasets.Dataset，Trainer 會退回自己對每一筆呼叫 len(item["input_ids"])
        # 來排序（PreferenceDataset.__getitem__ 回傳的就是這個 key，天生相容，不用額外
        # 接一個 length 欄位）。風險見 train_fold() 的 group_by_length 說明。
        group_by_length=group_by_length,
        report_to=[],
        logging_steps=10 if max_steps else 50,
        # 關掉 tqdm 進度條、強制用純文字 print 記錄 loss —— Kaggle Notebook 預設會用
        # rich/widget 進度條，那些輸出只進 __notebook__.ipynb 的 cell output，不會出現
        # 在 `kaggle kernels output` 抓得到的純文字 log，事後完全查不到 loss 曲線。
        disable_tqdm=True,
        seed=42,
    )
    if max_steps is not None:
        kwargs["max_steps"] = max_steps
    return TrainingArguments(**kwargs)


def predict_logits(trainer: Trainer, tokenizer, df: pd.DataFrame, max_len: int) -> np.ndarray:
    """對沒有 label 的 DataFrame（例如 test set）跑推論，回傳 (n, N_CLASSES) 的原始 logits
    （softmax 之前）——temperature scaling 要在這個尺度上配溫度，不能對已經 softmax
    過的機率配。
    """
    ds = PreferenceDataset(df, tokenizer, max_len, labels=None)
    return np.asarray(trainer.predict(ds).predictions)


def predict_probs(trainer: Trainer, tokenizer, df: pd.DataFrame, max_len: int) -> np.ndarray:
    """對沒有 label 的 DataFrame（例如 test set）跑推論，回傳 (n, N_CLASSES) 機率。"""
    return softmax(predict_logits(trainer, tokenizer, df, max_len))


def swap_ab(df: pd.DataFrame) -> pd.DataFrame:
    """回傳 response_a_text / response_b_text 對調後的複本，其餘欄位不變 ——
    PreferenceDataset 只讀這兩欄跟 prompt_text 建輸入，對調這兩欄就等於把
    response_a / response_b 的順序整個倒過來重新推論一次。
    """
    swapped = df.copy()
    swapped["response_a_text"] = df["response_b_text"].to_numpy()
    swapped["response_b_text"] = df["response_a_text"].to_numpy()
    return swapped


def train_fold(
    fold: int = 0,
    model_name: str = MODEL_NAME,
    n_folds: int = N_FOLDS,
    max_len: int = MAX_LEN,
    epochs: int = 2,
    batch_size: int = 8,
    lr: float = 2e-5,
    lr_scheduler_type: str = "linear",
    fp16: bool = True,
    output_dir: Path | None = None,
    max_train_rows: int | None = None,
    max_valid_rows: int | None = None,
    max_steps: int | None = None,
    eval_steps: int | None = None,
    eval_subset_rows: int | None = None,
    label_smoothing: float = 0.0,
    group_by_length: bool = False,
    ab_swap_prob: float = 0.0,
) -> dict:
    """練一個 fold，存權重，回傳 {"score", "n_nonfinite", "diverged", "output_dir",
    "trainer", "tokenizer"}。

    預設只練 fold 0，不是全部 n_folds —— 先確認贏過 baseline_prior.py 印出的分數，
    再決定要不要花時間跑滿整個 CV。

    `max_train_rows` / `max_valid_rows` / `max_steps` / `eval_steps` 是煙霧測試用的：
    隨機抽一小撮資料、跑幾十步就評估一次，把「資料→tokenize→forward→eval→存檔」整條
    路徑在幾分鐘內走過一遍，而不是每次改動都要賭一整個 epoch（30-60 分鐘 GPU 時間）
    才知道炸不炸。跑煙霧測試時務必把 `lr_scheduler_type` 設成 "constant_with_warmup"
    ——用預設的 "linear" 配上 `max_steps` 很小的話，學習率暖身完就立刻開始衰減，
    根本沒有停留在 peak LR 的時間，測不出「訓練到 peak LR 附近才發散」這種問題
    （這正是本專案第一次煙霧測試沒抓到、完整訓練卻在 peak LR 附近整個發散的原因）。

    `eval_subset_rows` 是完整訓練用的加速選項：訓練中途的週期性評估只在這個子集上跑
    （原本每次評估都對完整驗證集跑一次，實測光是評估就佔掉總訓練時間近一半），最後
    收斂完仍然會對完整驗證集重新 `evaluate()` 一次，回傳的 `score` 保證是完整驗證集
    的分數，不會被子集的雜訊污染。

    `label_smoothing`（milestone 3）：0.0 是關閉，跟原本行為一樣；HF Trainer 內建
    支援，不用自己改 labels 或 loss function。實測 0.1 讓 fold 0 valid log loss
    從 1.06840 變差成 1.08815，已經放棄，不要再試（見 README.md）。

    `group_by_length`：把長度相近的樣本分到同一個 batch，減少 padding 浪費。
    False 是關閉，跟原本行為一樣。風險：這正好會把最長的句子集中到同一批，獨立的
    profiling kernel 已經測過最壞情況（fold 0 最長 16 筆組成一個 batch）單步
    forward+backward 不會 OOM（餘裕 23.6%），但那是單步測試，完整一個 epoch 訓練
    下來記憶體碎片化累積會不會更緊繃還沒驗證過，第一次用這個設定時不要跳過
    `diverged`/`n_nonfinite` 的檢查。

    `ab_swap_prob`（milestone 3，訓練時 a/b 對調增強）：訓練集每一筆資料在每次
    被 `PreferenceDataset.__getitem__` 抓取時，有這個機率被動態對調
    response_a/response_b（連同標籤一起用 `swap_ab_label()` 對調：0⟷1，2 不變），
    每個 epoch 重新擲一次骰子，同一列在不同 epoch 可能拿到不同順序。只套用在
    訓練集，驗證集固定用原始順序不受影響，`score` 才能跟沒開這個選項的訓練直接
    比較。0.0 是關閉，跟原本行為一樣。動機：winner_tie 診斷（見
    `calibrate_folds.py`）發現模型對 `winner_model_a`/`winner_model_b` 兩類的
    原始（未做 TTA）log loss 落差很大（1.04710 vs 1.11257），推論時的 TTA 已經
    在事後修正一部分，這裡要測的是訓練時直接解決順序偏見，減少對推論時 TTA
    的依賴。

    `batch_size` 調大要非常小心：煙霧測試只能驗證穩定性（會不會發散），驗不出「完整
    資料集上的記憶體上限」——`max_train_rows` 抽樣的子集很難剛好抽到全是接近
    `max_len` 上限的最壞情況那幾批。實測 batch_size=16 在 3000 筆的煙霧測試上完全
    穩定，換成完整的 45746 筆卻在訓練中途 CUDA OOM（T4 記憶體只差 66MB）。batch_size
    調大之前，煙霧測試過關不代表完整資料集上安全。
    """
    # Kaggle 的 GPU kernel 預設給 T4 x2；HF Trainer 偵測到多張卡會自動包成
    # nn.DataParallel，這是已知會在 eval 階段的 predictions gather 上出怪問題的來源
    # （一個訊號：「gather along dimension 0 ... all input tensors were scalars」的
    # warning）。deberta-v3-base 在 batch_size 8 / max_len 512 下單張 T4 就跑得動，
    # 沒有 DP 帶來的好處，直接限制成單卡排除這個變因。用 setdefault 而不是強制覆蓋，
    # 呼叫端仍可自行指定 CUDA_VISIBLE_DEVICES。
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

    output_dir = Path(output_dir) if output_dir is not None else MODEL_DIR / f"fold{fold}"

    train = load_train()
    print(f"train: {len(train)} 列")
    train = add_folds(train, n_folds=n_folds)
    tr_idx, va_idx = fold_indices(train, fold)
    print(f"fold {fold}: train {len(tr_idx)} 列, valid {len(va_idx)} 列")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 分類頭（deberta-v3-base 本身沒有，from_pretrained 會隨機初始化一層新的）是在
    # Trainer 建立、TrainingArguments(seed=...) 生效之前就跑掉的——實測同一個 fold
    # 用完全相同的設定重跑，valid log loss 可以飄動 0.01~0.02（跟 label smoothing/
    # group_by_length 那兩次判定「有害」的差距同一個量級），才發現這裡才是真正決定
    # 起始點隨機性的地方，TrainingArguments 的 seed 只固定得了訓練「過程」（資料
    # 洗牌順序、dropout），固定不了「起點」。這裡先呼叫 set_seed() 才能讓同一個
    # seed 重跑兩次得到同一個分類頭初始值，A/B 比較才有意義。
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=N_CLASSES)
    # 實測 deberta-v3-base 在 HF Hub 上是用 fp16 存的，新版 transformers 的
    # from_pretrained 預設照抄 checkpoint 原本的 dtype，跟 TrainingArguments(fp16=False)
    # 完全無關 —— 結果是在跑「沒有 loss scaler 保護的裸 fp16 訓練」，梯度撐不了多久
    # 就溢位成 NaN，調低學習率只是延後發生、不是解法。強制轉 fp32 才是真正對應
    # fp16=False 的意圖。
    print(f"model 載入時的 dtype：{next(model.parameters()).dtype}")
    model = model.float()

    tr_df = train.iloc[tr_idx].reset_index(drop=True)
    va_df = train.iloc[va_idx].reset_index(drop=True)
    # 用隨機抽樣而不是頭幾列 —— 頭幾列在煙霧測試時永遠是同一批，測不到資料的多樣性。
    if max_train_rows is not None:
        tr_df = tr_df.sample(n=min(max_train_rows, len(tr_df)), random_state=SEED).reset_index(drop=True)
    if max_valid_rows is not None:
        va_df = va_df.sample(n=min(max_valid_rows, len(va_df)), random_state=SEED).reset_index(drop=True)
    tr_ds = PreferenceDataset(tr_df, tokenizer, max_len, tr_df["label"].to_numpy(), ab_swap_prob=ab_swap_prob)
    va_ds = PreferenceDataset(va_df, tokenizer, max_len, va_df["label"].to_numpy())

    # 訓練中途的週期性評估用子集（快很多），最後才對完整驗證集重新算一次真正的分數。
    if eval_subset_rows is not None and eval_subset_rows < len(va_df):
        va_df_periodic = va_df.sample(n=eval_subset_rows, random_state=SEED).reset_index(drop=True)
        va_ds_periodic = PreferenceDataset(va_df_periodic, tokenizer, max_len, va_df_periodic["label"].to_numpy())
    else:
        va_ds_periodic = va_ds

    stop_callback = StopOnNonFiniteLoss()
    trainer = Trainer(
        model=model,
        args=_training_args(
            output_dir, epochs, batch_size, lr, lr_scheduler_type=lr_scheduler_type,
            max_steps=max_steps, eval_steps=eval_steps, fp16=fp16,
            label_smoothing=label_smoothing, group_by_length=group_by_length,
        ),
        train_dataset=tr_ds,
        eval_dataset=va_ds_periodic,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=_compute_metrics,
        callbacks=[stop_callback],
    )
    trainer.train()
    if stop_callback.triggered:
        print("偵測到 loss/grad_norm 變成 NaN，已提前停止訓練 —— 這次的權重不可信，不要拿去推論")

    # 存檔緊接在 train() 後面、explicit evaluate() 之前 —— 不用 load_best_model_at_end
    # 挑歷史最佳（見 _training_args 的說明），trainer.model 現在就是訓練跑完當下的
    # 最終狀態，這裡先存起來，後面的 evaluate() 就算出狀況也不會白跑一整個 epoch 的
    # GPU 時間卻什麼都沒留下。
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    # TrainingArguments 的 output_dir 跟這裡的最終存檔目錄是同一個路徑，Trainer 自己
    # 的 save_steps 週期性存檔（含 optimizer/scheduler state，體積是模型本身的 2-3
    # 倍）還留在 output_dir/checkpoint-*/ 底下，跟上面剛存好的最終權重是同一份東西的
    # 重複備份。單一 fold 時這筆多餘的空間還在 Kaggle 磁碟額度內，5 個 fold 一次跑完
    # 疊起來就會把磁碟塞爆（實測踩到：5 folds 沒清、跑到一半磁碟就滿了）。權重已經
    # 存到 output_dir 頂層，這些子目錄可以直接刪掉。
    for checkpoint_dir in output_dir.glob("checkpoint-*"):
        shutil.rmtree(checkpoint_dir)
    print(f"模型已存到 {output_dir}（訓練中途的 checkpoint-* 已清除）")

    # 明確傳完整的 va_ds —— 訓練中途用的可能是子集，最終回報的分數必須是完整驗證集
    # 算出來的，不能被子集的雜訊污染。
    metrics = trainer.evaluate(eval_dataset=va_ds)
    score = metrics["eval_log_loss"]
    n_nonfinite = metrics.get("eval_n_nonfinite", 0)
    delta = UNIFORM_LOGLOSS - score
    # n_nonfinite > 0 代表分數是拿均勻機率湊出來的假象（例如全部 clamp 之後 delta 剛好
    # 等於 0，會被誤判成「打平基準」）——只要有非有限值，不管 delta 多少一律算沒過關。
    passed = n_nonfinite == 0 and not stop_callback.triggered and delta > 0
    print(f"\nvalid log loss   {score:.5f}")
    print(f"均勻亂猜基準      {UNIFORM_LOGLOSS:.5f}  (ln 3)")
    print(f"改善              {delta:+.5f}  {'✓ 優於基準' if passed else '✗ 未優於基準'}")
    if n_nonfinite:
        print(f"警告：{n_nonfinite}/{len(va_df)} 筆驗證預測是 NaN/inf，已夾到均勻機率計分 —— 分數不可信，先查訓練穩定性")

    return {
        "score": score,
        "n_nonfinite": n_nonfinite,
        "diverged": stop_callback.triggered,
        "output_dir": output_dir,
        "trainer": trainer,
        "tokenizer": tokenizer,
    }


def load_trained(output_dir: Path):
    """從已存的權重目錄載入 model + tokenizer（供推論 notebook 用，不需要 Trainer）。"""
    tokenizer = AutoTokenizer.from_pretrained(str(output_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(output_dir)).float()
    return model, tokenizer


def predict_logits_with_model(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """離線推論 notebook 用：不需要 Trainer / TrainingArguments，直接跑 forward，
    回傳 softmax 之前的原始 logits（temperature scaling 要配在這個尺度上）。
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    ds = PreferenceDataset(df, tokenizer, max_len, labels=None)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    all_logits = []
    with torch.no_grad():
        for start in range(0, len(ds), batch_size):
            batch = [ds[i] for i in range(start, min(start + batch_size, len(ds)))]
            inputs = collator(batch).to(device)
            logits = model(**inputs).logits.detach().cpu().numpy()
            all_logits.append(logits)
    return np.concatenate(all_logits, axis=0)


def predict_with_model(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """離線推論 notebook 用：不需要 Trainer / TrainingArguments，直接跑 forward。"""
    return softmax(predict_logits_with_model(model, tokenizer, df, max_len, batch_size))


def predict_logits_with_tta(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """a/b 對調 TTA，回傳原始順序與對調順序（已換回欄位對齊）logits 的平均。

    對付位置偏誤：模型可能學到偏好放在 A 或 B 位置本身，而不是回覆的品質。回傳
    logits（不是機率）是為了跟 temperature scaling 串接——要接著配溫度的話，必須
    先在 logits 尺度合併成一組，再對『合併後的 logits』配溫度，在機率層級平均、
    再對已經攤平過的機率配溫度會失真。單純只要 TTA、不接 calibration 的話，直接
    對這裡回傳的結果做 softmax 即可。
    """
    logits_orig = predict_logits_with_model(model, tokenizer, df, max_len, batch_size)
    logits_swapped = predict_logits_with_model(model, tokenizer, swap_ab(df), max_len, batch_size)
    return average_swapped(logits_orig, logits_swapped)


def predict_with_tta(model, tokenizer, df: pd.DataFrame, max_len: int, batch_size: int = 32) -> np.ndarray:
    """a/b 對調 TTA：原始順序跟對調順序各推論一次，換回欄位對齊後在機率層級平均。"""
    probs_orig = predict_with_model(model, tokenizer, df, max_len, batch_size)
    probs_swapped = predict_with_model(model, tokenizer, swap_ab(df), max_len, batch_size)
    return average_swapped(probs_orig, probs_swapped)


def predict_test_and_save(trainer_result: dict, name: str = "submission.csv") -> Path:
    """train_fold() 回傳值直接餵進來，對 test.csv 推論並寫出提交檔。"""
    test = load_test()
    probs = predict_probs(trainer_result["trainer"], trainer_result["tokenizer"], test, MAX_LEN)
    sub = build_submission(test["id"], probs)
    return save_submission(sub, name=name)
''',
    'training_safety.py': r'''"""訓練中途的「要不要緊急煞車」判斷邏輯，抽成不依賴 torch/transformers 的純函式
——可以在本機測試決策本身對不對，不用等 GPU 上真的發散一次才知道邏輯有沒有錯。

真正接到 HF Trainer 的地方在 llmcls/train.py 的 StopOnNonFiniteLoss。
"""

from __future__ import annotations

import math


def should_stop_for_nonfinite(
    logs: dict,
    consecutive_nonfinite_grad_norm: int,
    grad_norm_patience: int = 3,
) -> tuple[bool, int]:
    """回傳 (要不要喊停, 更新後的連續非有限 grad_norm 次數)。

    `loss` 本身變成非有限值代表訓練真的壞了（權重已經被非有限值污染，之後每一步
    都是壞的），一次就立刻停。`grad_norm` 偶爾出現非有限值不是同一回事——fp16
    混合精度訓練下，GradScaler 算出某一步的梯度真的溢位時會自動跳過那次更新、
    調低 scale factor 再繼續，這是設計上就會發生的正常現象，loss 本身仍然健康，
    不代表模型壞掉。實測踩過這個誤判：group_by_length 那次「發散」在觸發停止的
    那一行 loss 是 1.079（跟前面每一步一樣正常），只有 grad_norm 是 inf——判定
    「有害」其實是這個過度敏感的偵測邏輯誤觸發，不是真的訓練壞掉。改成只有連續
    `grad_norm_patience` 次 log 都非有限值（不是單一次）才當作真的卡住、喊停。
    """
    loss = logs.get("loss")
    if isinstance(loss, (int, float)) and not math.isfinite(loss):
        return True, consecutive_nonfinite_grad_norm

    grad_norm = logs.get("grad_norm")
    if isinstance(grad_norm, (int, float)) and not math.isfinite(grad_norm):
        consecutive_nonfinite_grad_norm += 1
    else:
        consecutive_nonfinite_grad_norm = 0

    return consecutive_nonfinite_grad_norm >= grad_norm_patience, consecutive_nonfinite_grad_norm
''',
    'tta.py': r'''"""a/b 對調 TTA（test-time augmentation）：對付位置偏誤。

模型可能學到偏好某個位置（A 或 B）本身，而不是純粹依回覆品質判斷。把 response_a /
response_b 對調後再推論一次，兩次結果換回原本的欄位對齊後平均，讓最終預測不受
位置影響。

純 numpy，不需要 torch，可以在本機測試（真正的兩次推論在 llmcls/train.py，
需要 torch/transformers）。
"""

from __future__ import annotations

import numpy as np

# LABEL_COLS 的欄位順序：[winner_model_a, winner_model_b, winner_tie]。
_SWAP_COLUMNS = [1, 0, 2]


def average_swapped(arr_orig: np.ndarray, arr_swapped: np.ndarray) -> np.ndarray:
    """把對調順序推論出來的結果換回原本的 a/b 欄位對齊，再跟原始順序的結果平均。

    `arr_swapped` 是把 response_a/response_b 對調後推論出來的，它的欄位順序是
    [P(目前放在 A 位置的贏), P(目前放在 B 位置的贏), P(tie)]——但「目前放在 A 位置」
    的其實是原本的 response_b，所以要先把前兩欄位換回來（`arr_swapped[:, [1, 0, 2]]`），
    才能跟 `arr_orig` 對齊平均。同一個操作對 logits（softmax 之前）跟機率（softmax
    之後）都適用，純粹是欄位重排 + 逐元素平均。
    """
    arr_orig = np.asarray(arr_orig)
    arr_swapped = np.asarray(arr_swapped)
    if arr_orig.shape != arr_swapped.shape:
        raise ValueError(f"形狀不一致：arr_orig {arr_orig.shape} vs arr_swapped {arr_swapped.shape}")
    aligned = arr_swapped[:, _SWAP_COLUMNS]
    return (arr_orig + aligned) / 2
''',
}

_pkg_dir = pathlib.Path("/kaggle/working/llmcls_src/src/llmcls")
_pkg_dir.mkdir(parents=True, exist_ok=True)
for _name, _content in _llmcls_files.items():
    (_pkg_dir / _name).write_text(_content, encoding="utf-8")

sys.path.insert(0, "/kaggle/working/llmcls_src/src")
print("llmcls bootstrapped:", sorted(p.name for p in _pkg_dir.glob("*.py")))

**不要猜掛載路徑**——跟 `infer_deberta.py` 一樣的坑，直接 glob 找全部 fold 的
checkpoint 實際在哪。Dataset 裡的檔名是攤平的 `fold{N}__檔名`（不是巢狀
`fold{N}/檔名`），先還原成 HF `from_pretrained()` 認得的巢狀資料夾。

In [ ]:
import pathlib
import shutil

from llmcls.config import MODEL_DIR

for _p in pathlib.Path("/kaggle/input").glob("**/fold*__model.safetensors"):
    fold_name, _ = _p.name.split("__", 1)
    dst = MODEL_DIR / fold_name
    if not dst.exists():
        dst.mkdir(parents=True)
        for _f in _p.parent.glob(f"{fold_name}__*"):
            _, filename = _f.name.split("__", 1)
            shutil.copy(_f, dst / filename)

_fold_checkpoints = {}
for p in sorted(pathlib.Path("/kaggle/input").glob("**/fold*/model.safetensors")):
    fold_num = int(p.parent.name.replace("fold", ""))
    _fold_checkpoints[fold_num] = p.parent
for p in sorted(MODEL_DIR.glob("fold*/model.safetensors")):
    fold_num = int(p.parent.name.replace("fold", ""))
    _fold_checkpoints.setdefault(fold_num, p.parent)

print("找到的 fold checkpoint：")
for fold_num in sorted(_fold_checkpoints):
    print(f"  fold {fold_num}: {_fold_checkpoints[fold_num]}")
if not _fold_checkpoints:
    print("/kaggle/input 底下的項目：", sorted(str(p) for p in pathlib.Path("/kaggle/input").iterdir()))
    raise FileNotFoundError("找不到任何 fold checkpoint —— 檢查 dataset_sources 是否正確接上權重 Dataset")

對每個 fold：重建它自己的驗證切分（`add_folds` / `fold_indices` 是同一套決定性
切分邏輯，重跑一次會得到一模一樣的切分）、載入它自己的 checkpoint、量測四種組合，
T 用這個 fold 自己的驗證集重新配。兩次推論（原始順序、對調順序）都只做一次，
存下原始 logits，後面的組合全部從這兩組 logits 算出來，不重複跑 forward。

In [ ]:
import torch

from llmcls.calibration import apply_temperature, fit_temperature
from llmcls.config import MAX_LEN, N_FOLDS
from llmcls.cv import add_folds, fold_indices
from llmcls.data import load_train
from llmcls.metrics import log_loss, softmax
from llmcls.train import load_trained, predict_logits_with_model, swap_ab
from llmcls.tta import average_swapped

train = load_train()
train = add_folds(train, n_folds=N_FOLDS)

per_fold = {}
for fold in sorted(_fold_checkpoints):
    _, va_idx = fold_indices(train, fold=fold)
    va_df = train.iloc[va_idx].reset_index(drop=True)
    labels = va_df["label"].to_numpy()

    model, tokenizer = load_trained(_fold_checkpoints[fold])

    logits_orig = predict_logits_with_model(model, tokenizer, va_df, max_len=MAX_LEN, batch_size=32)
    logits_swapped = predict_logits_with_model(model, tokenizer, swap_ab(va_df), max_len=MAX_LEN, batch_size=32)

    # 用完這個 fold 的模型就釋放 GPU 記憶體，避免 5 個 fold 依序跑、常駐記憶體疊加。
    del model
    torch.cuda.empty_cache()

    probs_orig = softmax(logits_orig)
    probs_swapped_aligned = softmax(logits_swapped)[:, [1, 0, 2]]

    baseline_loss = log_loss(labels, probs_orig)
    swapped_only_loss = log_loss(labels, probs_swapped_aligned)
    tta_loss = log_loss(labels, average_swapped(probs_orig, softmax(logits_swapped)))

    t_baseline = fit_temperature(logits_orig, labels)
    temp_loss = log_loss(labels, apply_temperature(logits_orig, t_baseline))

    combined_logits = average_swapped(logits_orig, logits_swapped)
    t_combined = fit_temperature(combined_logits, labels)
    tta_temp_loss = log_loss(labels, apply_temperature(combined_logits, t_combined))

    probs_tta_temp = apply_temperature(combined_logits, t_combined)

    per_fold[fold] = {
        "n_valid": len(va_df),
        "baseline": baseline_loss,
        "swapped_only": swapped_only_loss,
        "tta": tta_loss,
        "temperature": temp_loss,
        "t_baseline": t_baseline,
        "tta_temperature": tta_temp_loss,
        "t_combined": t_combined,
        # winner_tie 診斷用：留住這個 fold 的原始標籤跟兩種機率，之後要拼起來按
        # 真實類別（0=winner_model_a、1=winner_model_b、2=winner_tie，對應
        # llmcls.config.LABEL_COLS 的 argmax 編碼）分開算 log loss。
        "labels": labels,
        "probs_baseline": probs_orig,
        "probs_tta_temp": probs_tta_temp,
    }
    print(
        f"fold {fold}（{len(va_df)} 列驗證集）：baseline {baseline_loss:.5f}  "
        f"TTA {tta_loss:.5f}  temp(T={t_baseline:.3f}) {temp_loss:.5f}  "
        f"TTA+temp(T={t_combined:.3f}) {tta_temp_loss:.5f}"
    )

彙總 5 個 fold：每個組合的平均/標準差、T 值跨 fold 穩不穩定、TTA+temperature
相對 baseline 的改善是不是每個 fold 都成立。

In [ ]:
import numpy as np

metrics_keys = ["baseline", "swapped_only", "tta", "temperature", "tta_temperature"]
print("=== 5 folds 明細 ===")
header = f"{'fold':<6}" + "".join(f"{k:>16}" for k in metrics_keys)
print(header)
for fold in sorted(per_fold):
    r = per_fold[fold]
    print(f"{fold:<6}" + "".join(f"{r[k]:>16.5f}" for k in metrics_keys))

print("\n=== 跨 fold 平均 / 標準差 ===")
for k in metrics_keys:
    vals = np.array([per_fold[f][k] for f in per_fold])
    print(f"{k:<16}  mean={vals.mean():.5f}  std={vals.std():.5f}")

t_values = np.array([per_fold[f]["t_combined"] for f in per_fold])
print(f"\nT（TTA+temperature，每個 fold 各自配）：{list(np.round(t_values, 3))}")
print(f"  mean={t_values.mean():.3f}  std={t_values.std():.3f}")
if t_values.std() < 0.15:
    print(f"  T 在各 fold 間穩定，硬編碼 T≈{t_values.mean():.3f} 到 infer_deberta.py 算合理")
else:
    print("  T 在各 fold 間變動較大，硬編碼單一常數不夠穩，建議用跨 fold 平均值")

improvement = np.array([per_fold[f]["baseline"] - per_fold[f]["tta_temperature"] for f in per_fold])
n_improved = int((improvement > 0).sum())
print(f"\nTTA+temperature 相對 baseline 的改善（5 folds）：{list(np.round(improvement, 5))}")
print(f"平均改善 {improvement.mean():+.5f}  標準差 {improvement.std():.5f}  {n_improved}/{len(improvement)} 個 fold 有改善")

print(
    "\n注意：5 個 fold 共用同一套模型/訓練 recipe/資料分布，不是 5 個獨立實驗——"
    "一致改善代表這不是單一切分的巧合，不代表效果量必然完全轉移到隱藏測試集。"
)

## winner_tie 診斷：損失主要出在哪一類？

5-fold 交叉驗證下，每一列訓練資料剛好被當過一次（且僅一次）驗證集——把 5 個 fold
的驗證集預測全部拼在一起，等於用**全部** `train.csv` 做了一次公平的 held-out 檢驗
（每列都是某個 fold 訓練時沒看過的資料）。用這份拼起來的結果，照真實答案分成
`winner_model_a`（label=0）/`winner_model_b`（label=1）/`winner_tie`（label=2）
三組分別算 log loss，才能看出損失是不是集中在某一類——如果 tie 明顯比另外兩類差
很多，才值得投入 tie 專屬的處理（例如額外的類別權重、focal loss、或針對 tie 的
特徵工程）；如果三類其實差不多，這個方向就不值得做，milestone 3 的力氣該留給
別的候選手法。

同時列出 baseline（沒做 TTA/校準的原始模型）跟 TTA+temperature（`infer_deberta.py`
正式上線用的組合）兩種算法的每類別分數——這樣才看得出來 TTA+校準對三個類別是
平均改善，還是剛好特別修正了其中一類（例如 TTA 對調 a/b 順序，理論上只會影響
a/b 兩類的順序偏見，對 tie 這種「兩邊打平」的情況未必有幫助）。

In [ ]:
import numpy as np

from llmcls.config import LABEL_COLS

all_labels = np.concatenate([per_fold[f]["labels"] for f in sorted(per_fold)])
all_probs_baseline = np.concatenate([per_fold[f]["probs_baseline"] for f in sorted(per_fold)], axis=0)
all_probs_tta_temp = np.concatenate([per_fold[f]["probs_tta_temp"] for f in sorted(per_fold)], axis=0)

print(f"拼接後總列數：{len(all_labels)}（應該等於 train.csv 的總列數，5 個 fold 互不重疊、剛好覆蓋全部）")
print(f"整體 baseline log loss（拼接後重算，應該接近 5 個 fold baseline 的加權平均）："
      f"{log_loss(all_labels, all_probs_baseline):.5f}")
print(f"整體 TTA+temperature log loss（拼接後重算）："
      f"{log_loss(all_labels, all_probs_tta_temp):.5f}")

print("\n=== 按真實類別拆解（全部 5 folds 拼接後）===")
print(f"{'類別':<18}{'筆數':>8}{'佔比':>8}{'baseline':>12}{'TTA+temp':>12}{'改善':>10}")
per_class_summary = {}
for class_idx, class_name in enumerate(LABEL_COLS):
    mask = all_labels == class_idx
    n = int(mask.sum())
    loss_baseline = log_loss(all_labels[mask], all_probs_baseline[mask])
    loss_tta_temp = log_loss(all_labels[mask], all_probs_tta_temp[mask])
    per_class_summary[class_name] = {
        "n": n, "baseline": loss_baseline, "tta_temperature": loss_tta_temp,
    }
    print(
        f"{class_name:<18}{n:>8}{n / len(all_labels):>8.1%}"
        f"{loss_baseline:>12.5f}{loss_tta_temp:>12.5f}{loss_baseline - loss_tta_temp:>+10.5f}"
    )

worst_class = max(per_class_summary, key=lambda k: per_class_summary[k]["tta_temperature"])
print(
    f"\nTTA+temperature 之後，log loss 最高的類別是「{worst_class}」"
    f"（{per_class_summary[worst_class]['tta_temperature']:.5f}）——"
    "如果這個數字明顯高於另外兩類，才值得投入 tie 專屬的處理；"
    "三類接近的話，代表損失是平均分散在三類，不是 tie 特別難。"
)